# 06 — Evaluation, Class Imbalance & Decision Thresholds

The default class is not the majority class, so accuracy alone can be misleading.

This stage compares:

- ROC-AUC
- PR-AUC
- precision / recall
- F1
- balanced accuracy
- confusion matrices
- probability thresholds

The emphasis is on **how the model behaves when identifying the minority default class**.

## Load data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_curve,
)

DATA_PATH = Path("../data/synthetic/synthetic_early_modeling_base.csv")
df = pd.read_csv(DATA_PATH)

freq_days = {
    "Weekly": 7,
    "Bi-weekly": 14,
    "Monthly": 28,
}

df["pass_due_cycle_ratio"] = (
    df["early_max_overdue_days"]
    / df["frequency_name"].map(freq_days)
)

df["missed_installment_proportion"] = (
    df["early_missed_installment_count"]
    / df["prediction_installment"].clip(lower=1)
)

df["overdue_amount_proxy"] = (
    df["early_missed_installment_count"]
    * df["installment_amount"]
)

df["overdue_proportion"] = (
    df["overdue_amount_proxy"]
    / (
        df["prediction_installment"].clip(lower=1)
        * df["installment_amount"]
    )
).clip(0, 1)

df.shape

## 1. Check the class distribution

In [ ]:
class_distribution = (
    df["is_good_or_bad"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Good (0)", 1: "Default (1)"})
    .to_frame("count")
)

class_distribution["percentage"] = (
    class_distribution["count"] / len(df) * 100
).round(2)

class_distribution

The minority class is the modeling focus. A trivial majority-class classifier can have high accuracy while detecting few defaults, so later metrics should reflect minority-class performance.

## 2. Train/test split

In [ ]:
target = "is_good_or_bad"

numeric_features = [
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
    "early_missed_installment_count",
    "missed_installment_proportion",
    "early_max_consecutive_missed",
    "early_max_overdue_days",
    "pass_due_cycle_ratio",
    "early_recovery_delay_cycles",
    "overdue_proportion",
]

categorical_features = [
    "frequency_name",
    "product_group",
    "sector",
    "region",
]

X = df[numeric_features + categorical_features]
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

len(X_train), len(X_test), y_train.mean(), y_test.mean()

## 3. Logistic regression: unweighted vs class-weighted

In [ ]:
def make_pipeline(class_weight=None):
    preprocess = ColumnTransformer([
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ])

    return Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight=class_weight,
            random_state=42,
        )),
    ])

models = {
    "Unweighted logistic": make_pipeline(None),
    "Class-weighted logistic": make_pipeline("balanced"),
}

predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict_proba(X_test)[:, 1]

## 4. Compare probability-based metrics

In [ ]:
rows = []

for name, p in predictions.items():
    rows.append({
        "model": name,
        "ROC-AUC": roc_auc_score(y_test, p),
        "PR-AUC": average_precision_score(y_test, p),
        "default_rate": y_test.mean(),
    })

metric_comparison = pd.DataFrame(rows)
metric_comparison

PR-AUC is especially informative when the positive class is less frequent because it focuses on the precision/recall trade-off for the default class.

## 5. Threshold tuning

In [ ]:
def threshold_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "threshold": threshold,
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
    }

thresholds = np.arange(0.10, 0.91, 0.05)

weighted_probs = predictions["Class-weighted logistic"]

threshold_results = pd.DataFrame([
    threshold_metrics(y_test, weighted_probs, t)
    for t in thresholds
])

threshold_results

In [ ]:
plt.figure(figsize=(9, 5))

for metric in ["precision", "recall", "f1", "balanced_accuracy"]:
    plt.plot(
        threshold_results["threshold"],
        threshold_results[metric],
        marker="o",
        label=metric
    )

plt.xlabel("Probability threshold")
plt.ylabel("Score")
plt.title("Classification performance across thresholds")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

A threshold of 0.5 is a convention, not a universal optimum.

For an operational default-warning system, the threshold should eventually reflect the relative cost of missed defaults versus unnecessary interventions.

## 6. Confusion matrix at a selected threshold

In [ ]:
selected_threshold = 0.50

weighted_pred = (
    weighted_probs >= selected_threshold
).astype(int)

cm = confusion_matrix(y_test, weighted_pred)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Good", "Default"]
).plot(values_format="d")

plt.title(f"Confusion matrix — threshold {selected_threshold:.2f}")
plt.tight_layout()
plt.show()

threshold_metrics(
    y_test,
    weighted_probs,
    selected_threshold
)

## 7. Precision-recall and ROC curves

In [ ]:
precision, recall, _ = precision_recall_curve(
    y_test,
    weighted_probs
)

fpr, tpr, _ = roc_curve(
    y_test,
    weighted_probs
)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Evaluation notes

The evaluation stage separates three questions:

**Ranking:** Can the model distinguish higher-risk loans from lower-risk loans?

→ ROC-AUC / PR-AUC

**Classification:** Which loans are actually flagged at a chosen threshold?

→ precision / recall / F1 / confusion matrix

**Decision:** Where should the threshold be placed for the intended intervention?

→ threshold and cost analysis

The threshold should therefore not be selected from accuracy alone.

## 9. Link to the real experimentation

The original modeling notebooks reported AUC, AUCPR, log loss, F1, precision and recall, and several experiments exposed how class balance and the chosen threshold changed the resulting confusion matrix.

The public version keeps that evaluation philosophy but reproduces it with synthetic data.

Later work can add:

- cost-sensitive threshold selection;
- calibration;
- KS / Gini;
- bootstrap confidence intervals;
- time-based validation.